In [1]:
%pip install pytesseract pillow opencv-python


Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-2.2.6-cp313-cp313-win_amd64.whl.metadata (60 kB)
   ---------------------------------------- 0.0/39.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/39.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/39.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/39.0 MB ? eta -:--:--
    --------------------------------------- 0.5/39.0 MB 634.9 kB/s eta 0:01:01
    --------------------------------------- 0.8/39.0 MB 829.0 kB/s eta 0:00:47
   - -------------------------------------- 1.0/39.0 MB 1.0 MB/s eta 0:00:38
   - -------------------------------------- 1.3/39.0 MB 1.1 MB/s eta 0:00:34
   - -------------------------------------- 1.6/39.0 MB 1.1 MB/s eta 0:00:35
   -- ------------------------------------- 2.1/39.0 MB 1.2 MB/s eta 0:00:31
   -- ------------------------------------- 2.1/39.0 MB 1.2 MB/s eta 0:00:31
   -- -----------------------

  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install dateparser

Defaulting to user installation because normal site-packages is not writeable

   -------------------- ------------------- 1/2 [dateparser]
   -------------------- ------------------- 1/2 [dateparser]
   -------------------- ------------------- 1/2 [dateparser]
   -------------------- ------------------- 1/2 [dateparser]
   -------------------- ------------------- 1/2 [dateparser]
   -------------------- ------------------- 1/2 [dateparser]
   -------------------- ------------------- 1/2 [dateparser]
   -------------------- ------------------- 1/2 [dateparser]
   -------------------- ------------------- 1/2 [dateparser]
   -------------------- ------------------- 1/2 [dateparser]
   ---------------------------------------- 2/2 [dateparser]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import cv2
import pytesseract
import re
import pandas as pd
import dateparser

# If on Windows, set the tesseract path
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# Load invoice image
image_path = '../machine-learning-projects/invoice-template-other.jpg'  # Replace with your invoice file
image = cv2.imread(image_path)
print(image)

[[[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 [[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 [[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 ...

 [[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 [[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 [[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [255 255 255]
  [255 255 255]
  [255 255 255]]]


https://github.com/UB-Mannheim/tesseract/wiki

In [20]:
# Convert to grayscale
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Optional: Thresholding for better OCR
gray = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)[1]

# OCR: Extract text
text = pytesseract.image_to_string(gray)
text = text.replace("\n", " ")  # Remove newlines for easier regex

print("------ OCR Text ------")
print(text)

------ OCR Text ------
Service Provider  8150 McGavock Pk, Nashvile, Tennessee. 37214. US. (855) 555-5565 | he la@ se-vicep-ovider.com  RECIPIENT: Invoice #1058  Nicole Santos  Issued 2024-02-19 1 Thans Way Nashville, Tennessee 37213 Due 2024-03-19 Total $381.40  For Services Rendered  PRODUCT / SERVICE DESCRIPTION UNIT TOTAL PRIGE Materials Required materials for completing 1 300.00 $300.00 service  Labour Hourly rate for labor of 2 technicians 8 60.00 $480.00 ‘Thanks for your business! Subtotal $780.00 Tax Rate $101.40  (13%) Total $881.40  POWERED BY  (@ JOBBER 


In [21]:
# ----------------------------
# Extract Invoice Number
# ----------------------------
invoice_match = re.search(r'INVOICE\s*#\s*([A-Za-z0-9-]+)', text, re.IGNORECASE)
invoice_number = invoice_match.group(1) if invoice_match else "Not Found"

# ----------------------------
# Extract Invoice Date
# ----------------------------
# OCR may have weird formats, try to find sequences of digits for dates
date_match = re.search(r'INVOICE DATE\s*([0-9]{6,8})', text, re.IGNORECASE)
if date_match:
    raw_date = date_match.group(1)
    # parse using dateparser
    invoice_date = dateparser.parse(raw_date).date() if dateparser.parse(raw_date) else raw_date
else:
    invoice_date = "Not Found"

# ----------------------------
# Extract Total Amount
# ----------------------------
total_match = re.search(r'TOTAL\s*\$?([\d,]+\.\d{2})', text, re.IGNORECASE)
total_amount = total_match.group(1) if total_match else "Not Found"

# ----------------------------
# Show results
# ----------------------------
print("Invoice Number:", invoice_number)
print("Invoice Date:", invoice_date)
print("Total Amount:", total_amount)


Invoice Number: 1058
Invoice Date: Not Found
Total Amount: 381.40
